<a href="https://colab.research.google.com/github/chadallen/insider_trading_detection/blob/main/Detect_insider_trading_on_prediction_markets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Polymarket Insider Trading Detector

**Links:** [GitHub](https://github.com/chadallen/insider_trading_detection) · [Google Drive (data)](https://drive.google.com/drive/folders/1rD3SXjFoLfCmlQOeftVj1LI-pbEu1o1u?usp=sharing) · [Dashboard](https://polymarket-dashboard-roan.vercel.app/)

---

## What this does

This notebook looks for suspicious trading patterns in resolved Polymarket prediction markets using two independent signals:

**1. Price-based scoring** (free — uses Polymarket's public API)
- VPIN — measures order flow imbalance (adapted from Easley, López de Prado & O'Hara, 2011)
- Time-weighted VPIN — same, but weighted toward moves closer to resolution
- Resolution surprise — how wrong was the market's implied probability vs. what actually happened?
- Late move ratio — what % of total price movement happened right before resolution?
- All four signals are fed into an **Isolation Forest** (unsupervised anomaly detector) to rank markets

**2. Wallet-based scoring** (costs Dune credits — run selectively)
- New wallet ratio — did fresh wallets appear right before resolution?
- Burst score — did any wallet place a suspicious number of trades in a short window?
- Directional consensus — was trading overwhelmingly one-sided?
- Trade VPIN — real order flow imbalance from actual on-chain trade data

The two scores are averaged into a **combined suspicion score** and pushed to GitHub, where the dashboard reads them live.

**Limitations:**
- Polymarket only gives 12h price granularity for closed markets — VPIN is approximated from price movement, not actual order flow
- Political markets have no price history via CLOB API (data retention wall) — wallet scoring only
- ~130 markets is a small sample; model is unsupervised (no labeled training data yet)

---

## Setup (first time / forked notebook)

1. **Update Cell 0** with your GitHub repo and Drive folder name
2. **Add Colab Secrets** (click the key icon in the left sidebar):
   - `DUNE_API_KEY` — from [dune.com](https://dune.com) (free tier works)
   - `GITHUB_TOKEN` — GitHub personal access token with `repo` scope
3. Make sure your GitHub repo has an `outputs/` folder

---

## Run order guide

### Normal session (data already saved to Drive)
If Cell 2 shows all checkpoints as loaded, you can skip the expensive re-fetches:
```
Cell 0 → Cell 1 → Cell 2 (load Drive)
→ Cell 6 (re-score with Isolation Forest)
→ Cell 11 (define wallet scoring rules)
→ Cell 14 (combine scores)
→ Cell 15 (save) → Cell 16 (push to GitHub)
```
⏱ ~2 min · 0 Dune credits

### Refresh price scores (free)
Run when you want to pick up newly resolved markets:
```
Cell 0 → Cell 1 → Cell 2
→ Cell 3 (fetch markets) → Cell 4 (fetch price histories, ~5 min)
→ Cell 5 (compute features) → Cell 6 (Isolation Forest)
→ Cell 11 → Cell 13 (re-run Dune for new top 15) → Cell 14
→ Cell 15 → Cell 16
```
⏱ ~10 min · ~4 Dune credits (Cell 13 only)

### Full refresh (run sparingly)
Run when adding new labeled cases or Drive is empty:
```
All cells in order: Cell 0 through Cell 16
```
⏱ ~20 min · ~8 Dune credits

### When to run the expensive Dune cells

| Situation | Cell 9 (labeled markets) | Cell 13 (top 15) |
|-----------|:---:|:---:|
| Added new labeled cases to Cell 8 | ✅ Yes | — |
| Top 15 changed after price refresh | — | ✅ Yes |
| Just refreshing the dashboard | — | ✅ Yes |
| Drive pkl files missing | ✅ Yes | ✅ Yes |
| Routine weekly refresh | — | ✅ Yes |

**Dune free tier:** ~1500 credits/month. Each query costs ~4 credits.


In [1]:
#@title Cell 0 — User Configuration
# ╔══════════════════════════════════════════════════════════════════╗
# ║  EDIT THIS CELL BEFORE RUNNING ANYTHING ELSE                   ║
# ║  If you forked this notebook, update the values below.         ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── GitHub settings ───────────────────────────────────────────────
# Format: "username/repo-name"
# Your repo needs an outputs/ folder (create one with a placeholder file)
GITHUB_REPO   = "chadallen/insider_trading_detection"
GITHUB_BRANCH = "main"

# ── Google Drive settings ──────────────────────────────────────────
# Folder name in your Google Drive root for pkl checkpoints
DRIVE_FOLDER  = "insider_trading_poc"

# ── API keys ───────────────────────────────────────────────────────
# Do NOT paste keys here. Add them as Colab Secrets instead:
#   1. Click the key icon in the left sidebar
#   2. Add secret: DUNE_API_KEY   (from dune.com, free tier works)
#   3. Add secret: GITHUB_TOKEN   (github.com/settings/tokens, repo scope)

# ── Data settings ─────────────────────────────────────────────────
DAYS_BACK      = 548   # how far back to look for resolved markets
TOP_N_MARKETS  = 15    # how many markets to fetch wallet data for in Cell 13

print("✅ Configuration loaded")
print(f"   GitHub repo:   {GITHUB_REPO}")
print(f"   Drive folder:  {DRIVE_FOLDER}")
print(f"   Days back:     {DAYS_BACK}")
print(f"   Top N markets: {TOP_N_MARKETS}")

✅ Configuration loaded
   GitHub repo:   chadallen/insider_trading_detection
   Drive folder:  insider_trading_poc
   Days back:     548
   Top N markets: 15


In [2]:
#@title Cell 1 — Install dependencies and imports
# Run once per session. All core imports live here so any cell can be run independently.

!pip install requests pandas numpy scikit-learn plotly -q
!pip install PyGithub -q

import requests
import pandas as pd
import numpy as np
import json
import os
import pickle
import time
import io
from datetime import datetime, timedelta, timezone
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

print("✅ Dependencies installed and imports loaded")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.5 MB/s eta 0:00:00
✅ Dependencies installed and imports loaded


In [3]:
#@title Cell 2 — Mount Google Drive and load saved data
# Loads pkl checkpoints so you can skip expensive re-fetches.
# Variables that load as None will be recomputed when their cell runs.

from google.colab import drive
import os, pickle

try:
    drive.mount('/content/drive')
except ValueError:
    print("Drive already mounted ✅")

SAVE_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}\n")

def load_checkpoint(name):
    path = f"{SAVE_DIR}/{name}.pkl"
    if os.path.exists(path):
        with open(path, "rb") as f:
            data = pickle.load(f)
        print(f"  ✅ Loaded {name}")
        return data
    else:
        print(f"  ⚠️  Not found: {name} — will be computed when its cell runs")
        return None

print("Loading saved data...\n")
df_markets    = load_checkpoint("df_markets")
df_scored     = load_checkpoint("df_scored")
histories     = load_checkpoint("histories")
dune_results  = load_checkpoint("dune_results")
df_wallet_agg = load_checkpoint("df_wallet_agg")
df_combined   = load_checkpoint("df_combined")

print("\nDone.")
print("If all loaded ✅, you can skip to Cell 6 (re-score) or Cell 13 (refresh top 15 wallet data).")

Mounted at /content/drive
Save directory: /content/drive/MyDrive/insider_trading_poc

Loading saved data...

  ✅ Loaded df_markets
  ✅ Loaded df_scored
  ✅ Loaded histories
  ✅ Loaded dune_results
  ✅ Loaded df_wallet_agg
  ✅ Loaded df_combined

Done.
If all loaded ✅, you can skip to Cell 6 (re-score) or Cell 13 (refresh top 15 wallet data).


---
## Section A: Price-Based Scoring

**Cells 3–6.** Uses Polymarket's free public API. No API keys required.

**Skip to Cell 6** if `df_scored` loaded from Drive — it will re-run the Isolation Forest on the saved features.

**Run Cells 3–6** if:
- `df_scored` is None (not found on Drive)
- You want to pick up newly resolved markets
- It has been more than a week since your last run

> Cell 4 fetches price history for ~130 markets and takes 3–5 minutes.

In [7]:
#@title Cell 3 — Fetch resolved markets
# Fetches resolved markets across Geopolitics, Big Tech, and Culture tags.
# Uses /events endpoint (markets are nested inside events) with one call per tag.
# Takes top 250 markets by volume per tag, then deduplicates across tags.
# Result stored in df_markets.
# ⏱ ~10 seconds · 0 credits

import requests
import pandas as pd
import json
from datetime import datetime, timedelta, timezone

TAG_IDS = {
    "geopolitics": 100264,
    "big_tech":    1401,
    "culture":     596,
}

TOP_N_PER_TAG = 250  # top markets by volume within each tag

def fetch_markets_for_tag(tag_id, tag_name, days_back=1000, limit=200):
    end_date = datetime.now(timezone.utc)
    start_date = end_date - timedelta(days=days_back)
    url = "https://gamma-api.polymarket.com/events"
    params = {
        "tag_id":       tag_id,
        "closed":       "true",
        "limit":        limit,
        "after":        start_date.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "order":        "endDate",
        "ascending":    "false",
        "related_tags": "true",
    }
    response = requests.get(url, params=params)
    response.raise_for_status()

    markets = []
    for event in response.json():
        for m in event.get("markets", []):
            raw_tokens = m.get("clobTokenIds", "[]")
            try:
                tokens = json.loads(raw_tokens) if isinstance(raw_tokens, str) else raw_tokens
                token_id = tokens[0] if tokens else None
            except:
                token_id = None
            markets.append({
                "market_id":       m.get("conditionId"),
                "token_id":        token_id,
                "question":        m.get("question"),
                "resolution_time": m.get("endDate"),
                "final_price":     m.get("outcomePrices"),
                "volume":          float(m.get("volume") or 0),
                "category":        tag_name,
            })

    df = pd.DataFrame(markets)
    df = df[df["token_id"].notna() & (df["volume"] > 0)]

    # Take top N by volume within this tag
    df = df.nlargest(TOP_N_PER_TAG, "volume").reset_index(drop=True)
    print(f"  {tag_name}: {len(df)} markets (top {TOP_N_PER_TAG} by volume)")
    return df

print(f"Fetching resolved markets for {len(TAG_IDS)} tags...\n")
frames = []
for tag_name, tag_id in TAG_IDS.items():
    df = fetch_markets_for_tag(tag_id, tag_name, days_back=DAYS_BACK)
    frames.append(df)

df_markets = pd.concat(frames, ignore_index=True)

# Deduplicate — a market can appear under multiple tags
before = len(df_markets)
df_markets = df_markets.drop_duplicates(subset="market_id").reset_index(drop=True)
dupes = before - len(df_markets)

print(f"\n✅ {len(df_markets)} unique markets ({dupes} duplicates removed)")
print(f"   Category breakdown:")
print(df_markets["category"].value_counts().to_string())
print(f"\n{df_markets[['question', 'category', 'volume']].head(10).to_string()}")

Fetching resolved markets for 3 tags...

  geopolitics: 56 markets (top 250 by volume)
  big_tech: 250 markets (top 250 by volume)
  culture: 250 markets (top 250 by volume)

✅ 544 unique markets (12 duplicates removed)
   Category breakdown:
category
big_tech       250
culture        238
geopolitics     56

                                                     question     category         volume
0                     Trump positive favorability on Day 100?  geopolitics  327817.992944
1            Trump positive favorability on inauguration day?  geopolitics  327610.198338
2                  Trump flips Kamala on 538 before election?  geopolitics  248753.700439
3                           Will Trump lead in RCP on Oct 18?  geopolitics  223269.524993
4                 Trump Silver Bulletin odds >50% on Friday?   geopolitics  207102.888486
5           Will Kamala lead in RCP by 3.5 or more on Oct 18?  geopolitics  204662.423122
6                Will Kamala lead in RCP by 1-1.4  on Oct 18

In [9]:
#@title Cell 4 — Fetch price histories
# Calls CLOB API for each market's price history.
# Closed markets only support 12h+ granularity (fidelity=720 minimum).
# Only keeps markets with starting price 0.15–0.85 (otherwise not contested).
# ⏱ 3–5 minutes · 0 credits

def fetch_price_history(token_id, resolution_time, hours_before=48):
    if isinstance(resolution_time, str):
        res_time = datetime.fromisoformat(resolution_time.replace("Z", "+00:00"))
    else:
        res_time = resolution_time
    start_time = res_time - timedelta(hours=hours_before)
    url    = "https://clob.polymarket.com/prices-history"  # Note: clob not gamma-api
    params = {
        "market":   token_id,
        "interval": "max",
        "fidelity": 720,  # 12h min for closed markets — undocumented limitation
        "startTs":  int(start_time.timestamp()),
        "endTs":    int(res_time.timestamp()),
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
        if not data or "history" not in data or not data["history"]:
            return pd.DataFrame()
        df = pd.DataFrame(data["history"])
        df = df.rename(columns={"t": "timestamp", "p": "price"})  # API uses t/p not timestamp/price
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
        df["price"]     = df["price"].astype(float)
        return df
    except Exception:
        return pd.DataFrame()

print(f"Fetching price histories for {len(df_markets)} markets...")
histories = {}


for i, (_, row) in enumerate(df_markets.iterrows()):
    if i % 25 == 0: print(f"  {i}/{len(df_markets)}...")
    if row["resolution_time"] is None:
        continue
    history = fetch_price_history(row["token_id"], row["resolution_time"])
    if len(history) < 3: continue
    if not (0.15 <= history["price"].iloc[0] <= 0.85): continue
    histories[row["token_id"]] = history

print(f"\n✅ Cached price histories for {len(histories)} markets")

Fetching price histories for 544 markets...
  0/544...
  25/544...
  50/544...
  75/544...
  100/544...
  125/544...
  150/544...
  175/544...
  200/544...
  225/544...
  250/544...
  275/544...
  300/544...
  325/544...
  350/544...
  375/544...
  400/544...
  425/544...
  450/544...
  475/544...
  500/544...
  525/544...

✅ Cached price histories for 317 markets


In [10]:
#@title Cell 5 — Compute price features
# Computes VPIN, time-weighted VPIN, resolution surprise, late move ratio,
# price volatility, max single move, and total price move.
# Results stored in df_scored.
# ⏱ ~5 seconds · 0 credits

import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

def compute_vpin_from_prices(history_df):
    if len(history_df) < 3: return None
    prices = history_df["price"].values
    changes = np.diff(prices)
    buy  = np.where(changes > 0, changes, 0).sum()
    sell = np.where(changes < 0, -changes, 0).sum()
    total = buy + sell
    return abs(buy - sell) / total if total > 0 else 0.0

def compute_time_weighted_vpin(history_df):
    """VPIN with linear weights — moves closer to resolution count more."""
    if len(history_df) < 3: return None
    prices  = history_df["price"].values
    changes = np.diff(prices)
    n       = len(changes)
    weights = np.arange(1, n + 1, dtype=float)  # 1, 2, 3...N
    w_buy   = np.where(changes > 0, changes * weights, 0).sum()
    w_sell  = np.where(changes < 0, -changes * weights, 0).sum()
    total   = w_buy + w_sell
    return abs(w_buy - w_sell) / total if total > 0 else 0.0

def compute_price_features(history_df):
    if len(history_df) < 2: return None
    prices  = history_df["price"].values
    changes = np.abs(np.diff(prices))
    return {
        "price_volatility": changes.std() if len(changes) > 1 else 0,
        "max_single_move":  changes.max() if len(changes) > 0 else 0,
        "final_price":      prices[-1],
        "starting_price":   prices[0],
        "total_price_move": abs(prices[-1] - prices[0]),
    }

def compute_resolution_surprise(history_df):
    if len(history_df) < 2: return None, None
    prices           = history_df["price"].values
    actual           = 1.0 if prices[-1] > 0.5 else 0.0
    surprise         = abs(actual - prices[0])
    total_move       = abs(prices[-1] - prices[0])
    final_step       = abs(prices[-1] - prices[-2])
    late_move_ratio  = (final_step / total_move) if total_move > 0.01 else 0.0
    return surprise, late_move_ratio

results = []
for _, row in df_markets.iterrows():
    history = histories.get(row["token_id"])
    if history is None or len(history) < 3: continue
    vpin      = compute_vpin_from_prices(history)
    tw_vpin   = compute_time_weighted_vpin(history)
    features  = compute_price_features(history)
    surprise, late_move = compute_resolution_surprise(history)
    if any(v is None for v in [vpin, tw_vpin, features, surprise, late_move]): continue
    results.append({
        "question":           row["question"],
        "volume":             row["volume"],
        "vpin_score":         vpin,
        "time_weighted_vpin": tw_vpin,
        "surprise_score":     surprise,
        "late_move_ratio":    late_move,
        **features,
    })

df_scored = pd.DataFrame(results)
print(f"✅ Computed features for {len(df_scored)} markets")
print(df_scored[["question", "vpin_score", "surprise_score", "late_move_ratio"]].head(5).to_string())

✅ Computed features for 317 markets
                                           question  vpin_score  surprise_score  late_move_ratio
0           Trump positive favorability on Day 100?    0.235423           0.485         0.003109
1  Trump positive favorability on inauguration day?    0.175588           0.600         0.000000
2        Trump flips Kamala on 538 before election?    0.627660           0.710         0.004944
3       Trump Silver Bulletin odds >50% on Friday?     0.340193           0.270         0.018904
4       Will Kamala lead in RCP by 2-2.4 on Oct 18?    0.297740           0.385         0.027487


In [17]:
#@title Cell 5b — Replace price-proxy VPIN with real volume-weighted VPIN from Dune [EXPENSIVE]
# Runs a single Dune query across all markets in df_scored.
# Overwrites vpin_score with real trade_vpin where data is available.
# Falls back to price-proxy vpin_score for markets not found in Dune.
#
# COSTS ~2 DUNE CREDITS. Re-run when:
#   - df_markets changes (new tags, new DAYS_BACK)
#   - You want fresher on-chain data
# Skip if vpin already patched this session.
# ⏱ ~2–4 min

def sql_quote(s): return "'" + s.replace("'", "''") + "'"

in_clause = ",\n    ".join(sql_quote(q) for q in df_scored["question"].tolist())

vpin_sql = f"""
WITH trades AS (
    SELECT question, price, amount
    FROM polymarket_polygon.market_trades
    WHERE question IN (
        {in_clause}
    )
),
vpin AS (
    SELECT
        question,
        SUM(CASE WHEN price > 0.5 THEN amount ELSE 0 END) AS yes_vol,
        SUM(CASE WHEN price <= 0.5 THEN amount ELSE 0 END) AS no_vol,
        SUM(amount) AS total_vol
    FROM trades
    GROUP BY question
)
SELECT
    question,
    ABS(yes_vol - no_vol) / NULLIF(total_vol, 0) AS trade_vpin,
    total_vol
FROM vpin
"""

print(f"── Submitting VPIN query for {len(df_scored)} markets ──────────────")
df_vpin, _ = run_dune_query(vpin_sql, label="vpin_all_markets", timeout=300)

if not df_vpin.empty:
    vpin_lookup = dict(zip(df_vpin["question"], df_vpin["trade_vpin"].astype(float)))

    before = len(df_scored)
    df_scored = df_scored[df_scored["question"].isin(vpin_lookup)].copy()
    df_scored["vpin_score"] = df_scored["question"].map(vpin_lookup)
    dropped = before - len(df_scored)

    print(f"\n  ✅ {len(df_scored)} markets with real VPIN data")
    print(f"  🗑️  {dropped} markets dropped (no Dune data)")

    print(f"\n── VPIN distribution after patch ───────────────────────────────")
    print(df_scored["vpin_score"].describe().round(4).to_string())
else:
    print("  ⚠️  Dune returned no results — df_scored unchanged")

print("\nRun Cell 6 to re-score with Isolation Forest using updated VPIN.")



── Submitting VPIN query for 317 markets ──────────────
  Execution ID: 01KKCYW5B8Z5KCGH7EJX8NC0AA
  Status: QUERY_STATE_EXECUTING — waiting 5s...
  💳 Cost: 0.9907 credits | Session total: 2.0662

  ✅ 313 markets with real VPIN data
  🗑️  4 markets dropped (no Dune data)

── VPIN distribution after patch ───────────────────────────────
count    313.0000
mean       0.7583
std        0.2067
min        0.0226
25%        0.6435
50%        0.8026
75%        0.9309
max        0.9996

Run Cell 6 to re-score with Isolation Forest using updated VPIN.


In [18]:
#@title Cell 6 — Score markets with Isolation Forest
# Runs unsupervised anomaly detection across 8 price features.
# contamination=0.1 → ~10% of markets flagged as anomalous.
# Adds 'suspicion_score' to df_scored (higher = more suspicious).
# Can be re-run on loaded data without re-fetching price histories.
# ⏱ ~2 seconds · 0 credits

feature_cols = [
    "vpin_score",         # order flow imbalance (price proxy)
    "time_weighted_vpin", # same, weighted toward resolution
    "volume",             # total USD traded
    "total_price_move",   # overall price change start→end
    "price_volatility",   # std dev of 12h price changes
    "max_single_move",    # largest single 12h price jump
    "surprise_score",     # how wrong was the market before resolution
    "late_move_ratio",    # % of total move that happened right before resolution
]

X        = df_scored[feature_cols].fillna(0)
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)
model    = IsolationForest(contamination=0.1, random_state=42)

df_scored["anomaly_score"]   = model.fit_predict(X_scaled)
df_scored["suspicion_score"] = -model.decision_function(X_scaled)

top = df_scored.nlargest(15, "suspicion_score")[
    ["question", "suspicion_score", "vpin_score", "surprise_score", "late_move_ratio", "volume"]
].reset_index(drop=True)

print(f"✅ Scored {len(df_scored)} markets\n")
print("Top 15 by price-based suspicion score:")
print(top.to_string())

✅ Scored 313 markets

Top 15 by price-based suspicion score:
                                                                                         question  suspicion_score  vpin_score  surprise_score  late_move_ratio        volume
0                         Will Elon Musk post 65-89 tweets from February 26 to February 28, 2026?         0.143534    0.416598           0.290        10.888889  3.568935e+05
1                                          Will there be 8-10 inches of snow in NYC this weekend?         0.135517    0.732601           0.175         4.301282  1.975819e+05
2   Will ARKO Petroleum Corp.'s market cap be between $800M and $850M at market close on IPO day?         0.113198    0.957138           0.590         1.441052  1.068187e+05
3                                                                          Will GTA 6 cost $100+?         0.108838    0.959975           0.165         0.000000  8.069449e+06
4                         Will MrBeast's next video get between 55 an

---
## Section B: Wallet-Based Scoring (Dune Analytics)

**Cells 7–14.** Queries on-chain trade data from `polymarket_polygon.market_trades`.

**Dune credits used:** ~4 per query. Free tier = ~20–30 credits/month.

| Cell | What it does | Credits | When to run |
|------|-------------|---------|-------------|
| Cell 7 | Define Dune API helpers | 0 | Always — just defines functions |
| Cell 8 | Load labeled cases | 0 | Always |
| Cell 9 | Fetch raw trades for labeled markets | ~4 | Only when adding new labeled cases |
| Cell 10 | Compute wallet features from raw data | 0 | After Cell 9 |
| Cell 11 | Define wallet suspicion scoring rules | 0 | Always — just defines a function |
| Cell 12 | Score labeled markets (validation) | 0 | After Cells 10 + 11 |
| **Cell 13** | **Aggregated query for top N markets** | **~4** | **When top 15 has changed** |
| Cell 14 | Combine price + wallet scores | 0 | After Cell 13 |

**If `dune_results` and `df_wallet_agg` loaded from Drive ✅:** Run Cell 7 (define helpers), Cell 11 (define rules), then jump to Cell 14.

In [14]:
#@title Cell 7 — Dune API helper functions
# Defines core helpers: execute_sql(), poll_until_done(), fetch_results()
# Credit helpers:       get_dune_credits(), get_execution_cost()
# High-level wrapper:   run_dune_query(sql, label)  ← use this in Cells 9 & 13
#
# Required by Cell 9 and Cell 13 — always run this cell first.
# ⏱ Instant · 0 credits

import time
from google.colab import userdata

DUNE_API_KEY = userdata.get('DUNE_API_KEY')
DUNE_HEADERS = {"X-Dune-Api-Key": DUNE_API_KEY, "Content-Type": "application/json"}

# ── Low-level helpers ────────────────────────────────────────────────────────

def execute_sql(sql):
    r = requests.post(
        "https://api.dune.com/api/v1/sql/execute",
        headers=DUNE_HEADERS, json={"sql": sql, "performance": "medium"}
    )
    r.raise_for_status()
    return r.json()["execution_id"]

def poll_until_done(execution_id, timeout=180, interval=5):
    start = time.time()
    while time.time() - start < timeout:
        r = requests.get(
            f"https://api.dune.com/api/v1/execution/{execution_id}/status",
            headers=DUNE_HEADERS
        )
        r.raise_for_status()
        state = r.json()["state"]
        if state == "QUERY_STATE_COMPLETED": return True
        if state in ("QUERY_STATE_FAILED", "QUERY_STATE_CANCELLED"):
            print(f"  Query failed: {state}")
            return False
        print(f"  Status: {state} — waiting {interval}s...")
        time.sleep(interval)
    print("  Timed out.")
    return False

def fetch_results(execution_id):
    r = requests.get(
        f"https://api.dune.com/api/v1/execution/{execution_id}/results",
        headers=DUNE_HEADERS
    )
    r.raise_for_status()
    rows = r.json().get("result", {}).get("rows", [])
    return pd.DataFrame(rows)

# ── Credit helpers ───────────────────────────────────────────────────────────

_session_credits_used = 0.0  # running total for this Colab session

def get_execution_cost(execution_id):
    """Return the credit cost of a completed execution (or None if unavailable)."""
    r = requests.get(
        f"https://api.dune.com/api/v1/execution/{execution_id}/status",
        headers=DUNE_HEADERS
    )
    r.raise_for_status()
    return r.json().get("execution_cost_credits")

# ── High-level wrapper ───────────────────────────────────────────────────────

def run_dune_query(sql, label="query", timeout=180):
    """
    Execute a Dune SQL query end-to-end with credit tracking.
    Returns (df, execution_id). df is empty DataFrame on failure.
    """
    global _session_credits_used
    try:
        exec_id = execute_sql(sql)
        print(f"  Execution ID: {exec_id}")
        if poll_until_done(exec_id, timeout=timeout):
            df = fetch_results(exec_id)
            cost = get_execution_cost(exec_id)
            if cost is not None:
                _session_credits_used += cost
            cost_str = f"{cost:.4f}" if cost is not None else "unknown"
            print(f"  💳 Cost: {cost_str} credits | Session total: {_session_credits_used:.4f}")
            return df, exec_id
        return pd.DataFrame(), exec_id
    except Exception as e:
        print(f"  ❌ Error in run_dune_query({label}): {e}")
        return pd.DataFrame(), None

print("✅ Dune helpers defined:")
print("   execute_sql(), poll_until_done(), fetch_results()")
print("   get_dune_credits(), get_execution_cost()")
print("   run_dune_query(sql, label)  ← high-level wrapper for Cells 9 & 13")

✅ Dune helpers defined:
   execute_sql(), poll_until_done(), fetch_results()
   get_dune_credits(), get_execution_cost()
   run_dune_query(sql, label)  ← high-level wrapper for Cells 9 & 13


---
## Section C: Labeled Data & Validation

**Cells 8–12.** Ground truth cases used to calibrate the wallet scoring model.

> **Note on missing price history:** Political markets have no CLOB price data due to Polymarket's data retention policy (CLOB only retains price data for a limited window after resolution). All scoring for labeled cases is therefore wallet-only.

In [10]:
#@title Cell 8 — Load labeled insider trading cases
# Hardcoded ground truth cases. Add new confirmed/suspected cases here.
# Labels: CONFIRMED = reported by credible source · SUSPECTED = anomalous, unconfirmed · POSSIBLE = not investigated
# ⏱ Instant · 0 credits

import io

LABELS_CSV = """market_question,label
"Will the US strike Iran by February 28, 2026?",CONFIRMED
"Will María Corina Machado win the Nobel Peace Prize in 2025?",CONFIRMED
"Will Nicolás Maduro be removed from office by January 31, 2026?",SUSPECTED
"US x Venezuela military engagement by January 15, 2026?",SUSPECTED
"Which crypto company will ZachXBT expose for insider trading?",SUSPECTED
"Khamenei out as Supreme Leader of Iran by January 31?",SUSPECTED
"Taylor Swift x Travis Kelce get married by December 31?",SUSPECTED
"Will The Fate of Ophelia by Taylor Swift be the #1 searched song on Google this year?",SUSPECTED
"Gemini 3.0 Flash released by December 31?",POSSIBLE
"""

df_labels = pd.read_csv(io.StringIO(LABELS_CSV))
print(f"Loaded {len(df_labels)} labeled cases")
print(df_labels[["market_question", "label"]].to_string())

Loaded 9 labeled cases
                                                                         market_question      label
0                                          Will the US strike Iran by February 28, 2026?  CONFIRMED
1                           Will María Corina Machado win the Nobel Peace Prize in 2025?  CONFIRMED
2                        Will Nicolás Maduro be removed from office by January 31, 2026?  SUSPECTED
3                                US x Venezuela military engagement by January 15, 2026?  SUSPECTED
4                          Which crypto company will ZachXBT expose for insider trading?  SUSPECTED
5                                  Khamenei out as Supreme Leader of Iran by January 31?  SUSPECTED
6                                Taylor Swift x Travis Kelce get married by December 31?  SUSPECTED
7  Will The Fate of Ophelia by Taylor Swift be the #1 searched song on Google this year?  SUSPECTED
8                                              Gemini 3.0 Flash released by D

In [21]:
#@title Cell 9 — Fetch labeled market trades from Dune [EXPENSIVE]
# Runs one aggregated SQL query per labeled market (1 aggregated row returned).
# Results saved to dune_results dict and persisted to Drive in Cell 15.
#
# COSTS ~4–5 DUNE CREDITS total. Skip if dune_results loaded from Drive in Cell 2.
# Re-run only when:
#   - Adding new labeled cases to Cell 8
#   - Drive pkl was missing or corrupted
# ⏱ ~2–5 min

LABELED_MARKET_CONFIGS = {
    "iran":               {"label": "CONFIRMED",  "question_filter": "question = 'US strikes Iran by February 28, 2026?'",                                                        "start": "2026-02-26", "end": "2026-02-28", "resolution": "2026-02-28T23:59:00"},
    "nobel":              {"label": "CONFIRMED",  "question_filter": "question = 'Will María Corina Machado win the Nobel Peace Prize in 2025?'",                                 "start": "2025-10-08", "end": "2025-10-11", "resolution": "2025-10-10T11:00:00"},
    "maduro":             {"label": "SUSPECTED",  "question_filter": "question LIKE '%Maduro%'",                                                                                  "start": "2026-01-28", "end": "2026-01-31", "resolution": "2026-01-31T00:00:00"},
    "venezuela_military": {"label": "SUSPECTED",  "question_filter": "question = 'US x Venezuela military engagement by January 15, 2026?'",                                      "start": "2026-01-01", "end": "2026-01-04", "resolution": "2026-01-03T00:00:00"},
    "zachxbt":            {"label": "SUSPECTED",  "question_filter": "question = 'Will Axiom be accused of insider trading?'",                                                    "start": "2026-02-24", "end": "2026-02-27", "resolution": "2026-02-26T23:59:00"},
    "khamenei":           {"label": "SUSPECTED",  "question_filter": "question = 'Khamenei out as Supreme Leader of Iran by January 31?'",                                        "start": "2026-01-28", "end": "2026-02-01", "resolution": "2026-01-31T00:00:00"},
    "taylor_wedding":     {"label": "SUSPECTED",  "question_filter": "question = 'Taylor Swift x Travis Kelce get married by December 31?'",                                      "start": "2025-12-28", "end": "2026-01-01", "resolution": "2025-12-31T23:59:00"},
    "alpha_raccoon":      {"label": "SUSPECTED",  "question_filter": "question = 'Will The Fate of Ophelia by Taylor Swift be the #1 searched song on Google this year?'",       "start": "2025-11-28", "end": "2025-12-05", "resolution": "2025-12-04T00:00:00"},
    "gemini":             {"label": "POSSIBLE",   "question_filter": "question = 'Gemini 3.0 Flash released by December 31?'",                                                   "start": "2025-12-10", "end": "2025-12-18", "resolution": "2025-12-17T00:00:00"},
}

def build_labeled_sql(config):
    res_ts = config["resolution"].replace("T", " ")
    return f"""
WITH trades AS (
    SELECT block_time, maker, price, amount, token_outcome_name
    FROM polymarket_polygon.market_trades
    WHERE {config["question_filter"]}
      AND block_time BETWEEN TIMESTAMP '{config["start"]}' AND TIMESTAMP '{config["end"]}'
),
market_stats AS (
    SELECT COUNT(*) AS trade_count, COUNT(DISTINCT maker) AS unique_wallets,
           SUM(amount) AS total_volume
    FROM trades
),
resolution_times AS (SELECT TIMESTAMP '{res_ts}' AS res_time),
new_wallets_12h AS (
    SELECT SUM(t.amount) AS new_wallet_volume_12h FROM trades t
    CROSS JOIN resolution_times r
    WHERE t.maker IN (
        SELECT maker FROM (
            SELECT maker, MIN(block_time) AS first_seen FROM trades GROUP BY maker
        ) fw WHERE fw.first_seen >= r.res_time - INTERVAL '12' HOUR
    )
),
new_wallets_6h AS (
    SELECT SUM(t.amount) AS new_wallet_volume_6h FROM trades t
    CROSS JOIN resolution_times r
    WHERE t.maker IN (
        SELECT maker FROM (
            SELECT maker, MIN(block_time) AS first_seen FROM trades GROUP BY maker
        ) fw WHERE fw.first_seen >= r.res_time - INTERVAL '6' HOUR
    )
),
burst AS (
    SELECT MAX(trades_in_hour) AS burst_score
    FROM (SELECT DATE_TRUNC('hour', block_time) AS h, COUNT(*) AS trades_in_hour
          FROM trades GROUP BY DATE_TRUNC('hour', block_time)) x
),
directional AS (
    SELECT MAX(ov) * 1.0 / NULLIF(SUM(ov), 0) AS directional_consensus
    FROM (SELECT token_outcome_name, SUM(amount) AS ov FROM trades GROUP BY token_outcome_name) x
),
vpin AS (
    SELECT ABS(yes_vol - no_vol) / NULLIF(yes_vol + no_vol, 0) AS trade_vpin
    FROM (SELECT SUM(CASE WHEN price > 0.5 THEN amount ELSE 0 END) AS yes_vol,
                 SUM(CASE WHEN price <= 0.5 THEN amount ELSE 0 END) AS no_vol
          FROM trades) x
)
SELECT ms.trade_count, ms.unique_wallets, ms.total_volume,
    COALESCE(nw12.new_wallet_volume_12h, 0) / NULLIF(ms.total_volume, 0) AS new_wallet_ratio_12h,
    COALESCE(nw6.new_wallet_volume_6h, 0)  / NULLIF(ms.total_volume, 0) AS new_wallet_ratio_6h,
    b.burst_score, d.directional_consensus, v.trade_vpin
FROM market_stats ms
CROSS JOIN new_wallets_12h nw12
CROSS JOIN new_wallets_6h nw6
CROSS JOIN burst b
CROSS JOIN directional d
CROSS JOIN vpin v
"""

dune_results = {}
total_markets = len(LABELED_MARKET_CONFIGS)

for i, (name, config) in enumerate(LABELED_MARKET_CONFIGS.items(), 1):
    print(f"\n── [{i}/{total_markets}] {name.upper()} ({config['label']}) ──────────────────────────────")
    sql = build_labeled_sql(config)
    df, exec_id = run_dune_query(sql, label=name)
    if not df.empty:
        dune_results[name] = df
        row = df.iloc[0]
        print(f"  ✅ trades={int(row.get('trade_count',0)):,}  wallets={int(row.get('unique_wallets',0)):,}  vol=${float(row.get('total_volume',0)):,.0f}")
    else:
        dune_results[name] = pd.DataFrame()
        print("  ⚠️  No results returned")

print(f"\n── Summary {'─'*40}")
for name, df in dune_results.items():
    status = f"{len(df)} row(s)" if not df.empty else "EMPTY"
    print(f"  {name}: {status}")
print("\nRun Cell 15 to save dune_results to Drive.")


── [1/9] IRAN (CONFIRMED) ──────────────────────────────
  Execution ID: 01KKCJ9FX056Y9EZYWTHZF2G6X
  Status: QUERY_STATE_PENDING — waiting 5s...
  💳 Cost: 0.3416 credits | Session total: 0.3416
  ✅ trades=39,598  wallets=6,417  vol=$13,092,872

── [2/9] NOBEL (CONFIRMED) ──────────────────────────────
  Execution ID: 01KKCJ9Q58JRYT74XTP7TF28VM
  Status: QUERY_STATE_PENDING — waiting 5s...
  💳 Cost: 0.0785 credits | Session total: 0.4201
  ✅ trades=8,198  wallets=1,399  vol=$2,036,435

── [3/9] MADURO (SUSPECTED) ──────────────────────────────
  Execution ID: 01KKCJ9YECW3B65F1FR39H0SGV
  Status: QUERY_STATE_PENDING — waiting 5s...
  💳 Cost: 0.8072 credits | Session total: 1.2273
  ✅ trades=8,155  wallets=2,132  vol=$227,561

── [4/9] VENEZUELA_MILITARY (SUSPECTED) ──────────────────────────────
  Execution ID: 01KKCJA5P1EETPN28YWBY4G2NA
  Status: QUERY_STATE_PENDING — waiting 5s...
  💳 Cost: 0.1741 credits | Session total: 1.4014
  ✅ trades=4,758  wallets=1,000  vol=$2,653,313

── [5/

In [12]:
#@title Cell 10 — Extract wallet features from aggregated Dune results
# Reads the 1-row-per-market aggregated results from Cell 9b (or Drive).
# Requires dune_results populated from Cell 2 (Drive) or Cell 9b.
# ⏱ Instant · 0 credits

MARKET_LABELS = {
    "iran":               "CONFIRMED",
    "nobel":              "CONFIRMED",
    "maduro":             "SUSPECTED",
    "venezuela_military": "SUSPECTED",
    "zachxbt":            "SUSPECTED",
    "khamenei":           "SUSPECTED",
    "taylor_wedding":     "SUSPECTED",
    "alpha_raccoon":      "SUSPECTED",
    "gemini":             "POSSIBLE",
}

# Column name from Cell 9b → feature key used by compute_wallet_suspicion_score()
COLUMN_MAP = {
    "new_wallet_ratio":     "new_wallet_ratio",    # 12h
    "new_wallet_ratio_6h":  "new_wallet_ratio_6h",
    "trade_vpin":           "trade_vpin",
    "burst_score":          "burst_score",
    "directional_consensus":"directional_consensus",
    "wallet_concentration": "wallet_concentration",
    "total_volume":         "total_volume",
    "unique_wallets":       "unique_wallets",
    "trade_count":          "trade_count",
}

MIN_VOLUME_USD = 1000  # ignore markets with negligible volume (e.g. taylor_wedding at $228)

print("── Wallet features extracted from aggregated results ────────────────\n")
wallet_features = {}

for name, label in MARKET_LABELS.items():
    df = dune_results.get(name)

    if df is None or len(df) == 0:
        print(f"⚠️  {name}: no data in dune_results — re-run Cell 9b\n")
        continue

    row = df.iloc[0]
    features = {col: row.get(src, 0) for src, col in COLUMN_MAP.items()}

    # Flag low-volume markets — scores aren't meaningful at tiny scale
    volume = features.get("total_volume", 0)
    low_volume = volume < MIN_VOLUME_USD

    wallet_features[name] = features

    flag = " ⚠️  LOW VOLUME" if low_volume else ""
    print(f"── {name.upper()} ({label}){flag}")
    print(f"   Volume:          ${volume:>14,.2f}   Wallets: {int(features['unique_wallets']):,}")
    print(f"   New wallet 12h:  {features['new_wallet_ratio']:>8.1%}   New wallet 6h: {features['new_wallet_ratio_6h']:.1%}")
    print(f"   Trade VPIN:      {features['trade_vpin']:>8.3f}   Burst score:   {int(features['burst_score'])}")
    print(f"   Dir. consensus:  {features['directional_consensus']:>8.1%}   Wallet conc.:  {features['wallet_concentration']:.1%}")
    print()

print(f"✅ Extracted features for {len(wallet_features)}/{len(MARKET_LABELS)} markets")
print(f"   (Markets below ${MIN_VOLUME_USD:,} volume are flagged but still included)")
print("\nRun Cell 11 to define scoring rules, then Cell 12 to validate.")

── Wallet features extracted from aggregated results ────────────────

── IRAN (CONFIRMED)
   Volume:          $ 13,092,871.72   Wallets: 6,417
   New wallet 12h:      0.0%   New wallet 6h: 0.0%
   Trade VPIN:         0.552   Burst score:   409
   Dir. consensus:     77.6%   Wallet conc.:  9.2%

── NOBEL (CONFIRMED)
   Volume:          $  2,036,434.68   Wallets: 1,399
   New wallet 12h:     79.8%   New wallet 6h: 17.0%
   Trade VPIN:         0.371   Burst score:   80
   Dir. consensus:     66.4%   Wallet conc.:  20.5%

── MADURO (SUSPECTED)
   Volume:          $    227,560.78   Wallets: 2,132
   New wallet 12h:      3.6%   New wallet 6h: 0.1%
   Trade VPIN:         0.937   Burst score:   63
   Dir. consensus:     33.6%   Wallet conc.:  30.8%

── VENEZUELA_MILITARY (SUSPECTED)
   Volume:          $  2,653,313.19   Wallets: 1,000
   New wallet 12h:     71.8%   New wallet 6h: 67.6%
   Trade VPIN:         0.939   Burst score:   45
   Dir. consensus:     81.5%   Wallet conc.:  37.2%

── ZAC

In [13]:
#@title Cell 11 — Define wallet suspicion scoring rules
# Rule-based scoring function calibrated from labeled market analysis.
# Each of 5 signals contributes up to 0.20 (max total = 1.0).
# Required by Cell 12 and Cell 13 — always run this cell.
# ⏱ Instant · 0 credits

def compute_wallet_suspicion_score(features):
    """
    Rule-based wallet suspicion score (0.0 – 1.0).

    Thresholds calibrated from confirmed cases:
      Nobel (CONFIRMED):   new_wallet_ratio_12h = 79.8%  — dominant signal
      Maduro (SUSPECTED):  trade_vpin = 0.937            — dominant signal
      ZachXBT (SUSPECTED): burst_score = 270             — dominant signal
      Iran (CONFIRMED):    burst_score = 409, dir_consensus = 77.6%
    """
    score = 0.0
    nwr = features.get("new_wallet_ratio", 0)
    if nwr > 0.50: score += 0.20
    elif nwr > 0.10: score += 0.10

    nwr_6h = features.get("new_wallet_ratio_6h", 0)
    if nwr_6h > 0.10: score += 0.20
    elif nwr_6h > 0.05: score += 0.10

    vpin = features.get("trade_vpin", 0)
    if vpin > 0.80: score += 0.20
    elif vpin > 0.60: score += 0.10

    burst = features.get("burst_score", 0)
    if burst > 200: score += 0.20
    elif burst > 50: score += 0.10

    dc = features.get("directional_consensus", 0)
    if dc > 0.70: score += 0.20
    elif dc > 0.55: score += 0.10

    return round(score, 2)

print("✅ compute_wallet_suspicion_score() defined")
print("   Thresholds: new_wallet_12h >50%=+0.20 >10%=+0.10")
print("              new_wallet_6h  >10%=+0.20  >5%=+0.10")
print("              trade_vpin    >0.80=+0.20 >0.60=+0.10")
print("              burst_score    >200=+0.20  >50=+0.10")
print("              dir_consensus  >70%=+0.20  >55%=+0.10")

✅ compute_wallet_suspicion_score() defined
   Thresholds: new_wallet_12h >50%=+0.20 >10%=+0.10
              new_wallet_6h  >10%=+0.20  >5%=+0.10
              trade_vpin    >0.80=+0.20 >0.60=+0.10
              burst_score    >200=+0.20  >50=+0.10
              dir_consensus  >70%=+0.20  >55%=+0.10


In [14]:
#@title Cell 12 — Score labeled markets (validation)
# Applies wallet scoring rules to all ground truth cases.
# Requires: wallet_features (Cell 10) + compute_wallet_suspicion_score (Cell 11)
# ⏱ Instant · 0 credits

MIN_VOLUME_USD = 1000  # matches Cell 10

print("── Wallet suspicion scores (labeled markets) ────────────────────\n")
print(f"{'Market':<22} {'Label':<11} {'Score':>6}  {'Dominant signal'}")
print("─" * 75)

scores_by_label = {"CONFIRMED": [], "SUSPECTED": [], "POSSIBLE": []}

for name, features in wallet_features.items():
    label = MARKET_LABELS[name]
    volume = features.get("total_volume", 0)
    low_vol = volume < MIN_VOLUME_USD

    ws = compute_wallet_suspicion_score(features)
    scores_by_label[label].append(ws)

    # Identify the dominant signal for readability
    signals = {
        f"new_wallet_12h={features['new_wallet_ratio']:.0%}":  features["new_wallet_ratio"] * (0.20 if features["new_wallet_ratio"] > 0.50 else 0.10 if features["new_wallet_ratio"] > 0.10 else 0),
        f"new_wallet_6h={features['new_wallet_ratio_6h']:.0%}": features["new_wallet_ratio_6h"] * (0.20 if features["new_wallet_ratio_6h"] > 0.10 else 0.10 if features["new_wallet_ratio_6h"] > 0.05 else 0),
        f"trade_vpin={features['trade_vpin']:.3f}":             features["trade_vpin"] * (0.20 if features["trade_vpin"] > 0.80 else 0.10 if features["trade_vpin"] > 0.60 else 0),
        f"burst={int(features['burst_score'])}":                features["burst_score"] * (0.20 if features["burst_score"] > 200 else 0.10 if features["burst_score"] > 50 else 0),
        f"dir_consensus={features['directional_consensus']:.0%}": features["directional_consensus"] * (0.20 if features["directional_consensus"] > 0.70 else 0.10 if features["directional_consensus"] > 0.55 else 0),
    }
    dominant = max(signals, key=signals.get) if ws > 0 else "—"

    vol_flag = " ⚠️ low vol" if low_vol else ""
    print(f"{name:<22} {label:<11} {ws:>6.2f}  {dominant}{vol_flag}")

print("\n── Summary by label ─────────────────────────────────────────────\n")
for label, scores in scores_by_label.items():
    if scores:
        avg = sum(scores) / len(scores)
        print(f"  {label:<11} n={len(scores)}  avg={avg:.2f}  scores={[f'{s:.2f}' for s in scores]}")

print("\n── What the model caught ────────────────────────────────────────\n")
threshold = 0.40
caught = {name: compute_wallet_suspicion_score(f) for name, f in wallet_features.items()
          if compute_wallet_suspicion_score(f) >= threshold}
missed = {name: compute_wallet_suspicion_score(f) for name, f in wallet_features.items()
          if compute_wallet_suspicion_score(f) < threshold}

print(f"  At threshold {threshold:.2f}:")
print(f"  Flagged ({len(caught)}): {', '.join(caught.keys())}")
print(f"  Missed  ({len(missed)}): {', '.join(missed.keys()) or 'none'}")

── Wallet suspicion scores (labeled markets) ────────────────────

Market                 Label        Score  Dominant signal
───────────────────────────────────────────────────────────────────────────
iran                   CONFIRMED     0.40  burst=409
nobel                  CONFIRMED     0.60  burst=80
maduro                 SUSPECTED     0.30  burst=63
venezuela_military     SUSPECTED     0.80  trade_vpin=0.939
zachxbt                SUSPECTED     0.40  burst=270
khamenei               SUSPECTED     0.80  burst=63
taylor_wedding         SUSPECTED     0.50  dir_consensus=100% ⚠️ low vol
alpha_raccoon          SUSPECTED     0.80  dir_consensus=99%
gemini                 POSSIBLE      0.70  dir_consensus=97%

── Summary by label ─────────────────────────────────────────────

  CONFIRMED   n=2  avg=0.50  scores=['0.40', '0.60']
  SUSPECTED   n=6  avg=0.60  scores=['0.30', '0.80', '0.40', '0.80', '0.50', '0.80']
  POSSIBLE    n=1  avg=0.70  scores=['0.70']

── What the model caught ────

In [22]:
#@title Cell 13 — Aggregated Dune query for top N markets [EXPENSIVE]
# Runs a single SQL query returning 1 aggregated row per market.
#
# COSTS ~8 DUNE CREDITS. Re-run when:
#   - Top N changed significantly after a price score refresh
#   - df_wallet_agg is None (not on Drive)
#   - Refreshing the dashboard with fresh wallet data
#
# Skip this cell if df_wallet_agg loaded from Drive ✅ → jump to Cell 14.
# ⏱ ~2–3 min

top_n = df_scored.nlargest(TOP_N_MARKETS, "suspicion_score").copy().reset_index(drop=True)

def sql_quote(s): return "'" + s.replace("'", "''") + "'"
in_clause = ",\n    ".join(sql_quote(q) for q in top_n["question"].tolist())

# Use earliest resolution time in top_n as the date floor, with a 30-day buffer
top_n_with_dates = top_n.merge(
    df_markets[["question", "resolution_time"]],
    on="question", how="left"
)
date_floor = (pd.to_datetime(top_n_with_dates["resolution_time"]).min() - pd.Timedelta(days=30)).strftime("%Y-%m-%d")
print(f"  Date floor: {date_floor}")
aggregated_sql = f"""
WITH trades AS (
    SELECT question, block_time, maker, price, amount, token_outcome_name
    FROM polymarket_polygon.market_trades
    WHERE question IN ({in_clause})
      AND block_time >= TIMESTAMP '{date_floor}'
),
market_stats AS (
    SELECT question, COUNT(*) AS trade_count, COUNT(DISTINCT maker) AS unique_wallets,
           SUM(amount) AS total_volume, MAX(block_time) AS last_trade_time, MIN(block_time) AS first_trade_time
    FROM trades GROUP BY question
),
wallet_volumes AS (SELECT question, maker, SUM(amount) AS wallet_volume FROM trades GROUP BY question, maker),
top3 AS (
    SELECT question, SUM(wallet_volume) AS top3_volume
    FROM (SELECT question, wallet_volume, ROW_NUMBER() OVER (PARTITION BY question ORDER BY wallet_volume DESC) AS rn FROM wallet_volumes) r
    WHERE rn <= 3 GROUP BY question
),
resolution_times AS (SELECT question, MAX(block_time) AS inferred_resolution FROM trades GROUP BY question),
new_wallets_12h AS (
    SELECT t.question, SUM(t.amount) AS new_wallet_volume_12h FROM trades t
    JOIN resolution_times r ON t.question = r.question
    WHERE t.maker IN (
        SELECT maker FROM (SELECT maker, question, MIN(block_time) AS first_seen FROM trades GROUP BY maker, question) fw
        JOIN resolution_times rt ON fw.question = rt.question
        WHERE fw.first_seen >= rt.inferred_resolution - INTERVAL '12' HOUR AND fw.question = t.question
    ) AND t.question = r.question GROUP BY t.question
),
new_wallets_6h AS (
    SELECT t.question, SUM(t.amount) AS new_wallet_volume_6h FROM trades t
    JOIN resolution_times r ON t.question = r.question
    WHERE t.maker IN (
        SELECT maker FROM (SELECT maker, question, MIN(block_time) AS first_seen FROM trades GROUP BY maker, question) fw
        JOIN resolution_times rt ON fw.question = rt.question
        WHERE fw.first_seen >= rt.inferred_resolution - INTERVAL '6' HOUR AND fw.question = t.question
    ) AND t.question = r.question GROUP BY t.question
),
burst AS (
    SELECT question, MAX(trades_in_hour) AS burst_score
    FROM (SELECT question, DATE_TRUNC('hour', block_time) AS h, COUNT(*) AS trades_in_hour FROM trades GROUP BY question, DATE_TRUNC('hour', block_time)) x
    GROUP BY question
),
directional AS (
    SELECT question, MAX(ov) * 1.0 / NULLIF(SUM(ov), 0) AS directional_consensus
    FROM (SELECT question, token_outcome_name, SUM(amount) AS ov FROM trades GROUP BY question, token_outcome_name) x
    GROUP BY question
),
vpin AS (
    SELECT question, ABS(yes_vol - no_vol) / NULLIF(yes_vol + no_vol, 0) AS trade_vpin
    FROM (SELECT question, SUM(CASE WHEN price > 0.5 THEN amount ELSE 0 END) AS yes_vol, SUM(CASE WHEN price <= 0.5 THEN amount ELSE 0 END) AS no_vol FROM trades GROUP BY question) x
)
SELECT ms.question, ms.trade_count, ms.unique_wallets, ms.total_volume, ms.first_trade_time, ms.last_trade_time,
    t3.top3_volume, t3.top3_volume / NULLIF(ms.total_volume, 0) AS wallet_concentration,
    COALESCE(nw12.new_wallet_volume_12h, 0) / NULLIF(ms.total_volume, 0) AS new_wallet_ratio_12h,
    COALESCE(nw6.new_wallet_volume_6h, 0)  / NULLIF(ms.total_volume, 0) AS new_wallet_ratio_6h,
    b.burst_score, d.directional_consensus, v.trade_vpin
FROM market_stats ms
LEFT JOIN top3 t3              ON ms.question = t3.question
LEFT JOIN new_wallets_12h nw12 ON ms.question = nw12.question
LEFT JOIN new_wallets_6h nw6   ON ms.question = nw6.question
LEFT JOIN burst b              ON ms.question = b.question
LEFT JOIN directional d        ON ms.question = d.question
LEFT JOIN vpin v               ON ms.question = v.question
ORDER BY ms.question
"""

print(f"── Submitting aggregated Dune query ({len(top_n)} markets) ──────────────")
df_wallet_agg, _ = run_dune_query(aggregated_sql, label="top_n_markets", timeout=240)

if not df_wallet_agg.empty:
    print(f"\n── Results ({len(df_wallet_agg)} markets) ────────────────────────────────────────")
    cols = ["question", "unique_wallets", "new_wallet_ratio_12h", "new_wallet_ratio_6h", "burst_score", "trade_vpin"]
    print(df_wallet_agg[[c for c in cols if c in df_wallet_agg.columns]].to_string())
else:
    print("  ⚠️  No results returned — df_wallet_agg is empty")

  Date floor: 2024-08-09
── Submitting aggregated Dune query (15 markets) ──────────────
  Execution ID: 01KKCZG3NFJ5EA8V41JW7WGB7D
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_EXECUTING — waiting 5s...
  Status: QUERY_STATE_EXECUTING — waiting 5s...
  Status: QUERY_STATE_EXECUTING — waiting 5s...
  Status: QUERY_STATE_EXECUTING — waiting 5s...
  Query failed: QUERY_STATE_FAILED
  ⚠️  No results returned — df_wallet_agg is empty


In [22]:
#@title Cell 14 — Combine price + wallet scores
# Averages price-based (Cell 6) and wallet-based (Cell 13) scores.
# Requires: df_scored, df_wallet_agg, compute_wallet_suspicion_score.
# ⏱ Instant · 0 credits

top_n = df_scored.nlargest(TOP_N_MARKETS, "suspicion_score").copy().reset_index(drop=True)

print(f"── Combined scores (price + wallet) ────────────────────────────")
print(f"{'#':<3} {'Market':<50} {'Price':>7} {'Wallet':>7} {'Combined':>9}")
print("─" * 80)

combined = []
for i, row in top_n.iterrows():
    q           = row["question"]
    price_score = row["suspicion_score"]
    agg = df_wallet_agg[df_wallet_agg["question"] == q] if df_wallet_agg is not None and len(df_wallet_agg) > 0 else pd.DataFrame()

    if len(agg) > 0:
        a = agg.iloc[0]
        wallet_score = compute_wallet_suspicion_score({
            "new_wallet_ratio":      float(a["new_wallet_ratio_12h"] or 0),
            "new_wallet_ratio_6h":   float(a["new_wallet_ratio_6h"]  or 0),
            "trade_vpin":            float(a["trade_vpin"]            or 0),
            "burst_score":           float(a["burst_score"]           or 0),
            "directional_consensus": float(a["directional_consensus"] or 0),
        })
    else:
        wallet_score = None

    combined_score = (price_score + wallet_score) / 2 if wallet_score is not None else price_score
    combined.append({"question": q, "price_score": price_score, "wallet_score": wallet_score, "combined_score": combined_score})
    ws_str = f"{wallet_score:.3f}" if wallet_score is not None else "  N/A"
    print(f"{i+1:<3} {q[:50]:<50} {price_score:>7.3f} {ws_str:>7} {combined_score:>9.3f}")

df_combined = pd.DataFrame(combined).sort_values("combined_score", ascending=False).reset_index(drop=True)
print(f"\n── Top 5 by combined score ──────────────────────────────────────")
print(df_combined[["question", "price_score", "wallet_score", "combined_score"]].head().to_string())

── Combined scores (price + wallet) ────────────────────────────
#   Market                                               Price  Wallet  Combined
────────────────────────────────────────────────────────────────────────────────
1   Sentient FDV above $800M one day after launch?       0.119   0.500     0.309
2   Will Almanak FDV above $50M one day after launch?    0.111   0.500     0.305
3   Over $500K committed to the Sol Raffle public sale   0.085   0.700     0.392
4   Over $20M committed to the Space public sale?        0.079   0.700     0.389
5   Hyperlend FDV above $20M one day after launch?       0.063   0.700     0.382
6   Rainbow FDV above $30M one day after launch?         0.048   0.900     0.474
7   Trove FDV above $5M one day after launch?            0.043   0.700     0.371
8   Over $3M committed to the Infinex public sale?       0.028   0.500     0.264
9   Over $2M committed to the Infinex public sale?       0.013   0.500     0.257
10  Over $7M committed to the Fabric public 

---
## Section D: Save & Export

**Always run both cells** at the end of a session.

- **Cell 15** saves pkl checkpoints to Google Drive — lets you skip re-fetches next session
- **Cell 16** pushes CSVs to GitHub so the [dashboard](https://polymarket-dashboard-roan.vercel.app) updates

In [23]:
#@title Cell 15 — Save all data to Google Drive
# Always run at end of session to preserve work.
# dune_results is the large file (~28MB); others are small.
# ⏱ ~30 seconds · 0 credits

import pickle, os

SAVE_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(SAVE_DIR, exist_ok=True)

def save_checkpoint(obj, name):
    if obj is None:
        print(f"  ⚠️  Skipping {name} (is None)")
        return
    path = f"{SAVE_DIR}/{name}.pkl"
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"  ✅ Saved {name} ({os.path.getsize(path)/1024:.1f} KB)")

print(f"Saving to {SAVE_DIR}...\n")
save_checkpoint(df_markets,    "df_markets")
save_checkpoint(df_scored,     "df_scored")
save_checkpoint(histories,     "histories")
save_checkpoint(dune_results,  "dune_results")
save_checkpoint(df_wallet_agg, "df_wallet_agg")
save_checkpoint(df_combined,   "df_combined")

print(f"\nFiles in {SAVE_DIR}:")
for f in sorted(os.listdir(SAVE_DIR)):
    print(f"  {f} ({os.path.getsize(f'{SAVE_DIR}/{f}')/1024:.1f} KB)")

Saving to /content/drive/MyDrive/insider_trading_poc...

  ✅ Saved df_markets (44.9 KB)
  ✅ Saved df_scored (18.4 KB)
  ✅ Saved histories (157.5 KB)
  ✅ Saved dune_results (5.2 KB)
  ✅ Saved df_wallet_agg (3.9 KB)
  ✅ Saved df_combined (1.8 KB)

Files in /content/drive/MyDrive/insider_trading_poc:
  Polymarket Insider Trading Detector — Short Summary.gdoc (0.2 KB)
  df_combined.pkl (1.8 KB)
  df_markets.pkl (44.9 KB)
  df_scored.pkl (18.4 KB)
  df_wallet_agg.pkl (3.9 KB)
  dune_results.pkl (5.2 KB)
  histories.pkl (157.5 KB)


In [24]:
#@title Cell 16 — Export results to GitHub
# Pushes df_scored, df_wallet_agg, df_combined as CSVs to outputs/ in your GitHub repo.
# Dashboard reads df_combined.csv via GitHub raw URL.
# If file content is unchanged, no new commit is created — that is not an error.
# ⏱ ~10 seconds · 0 credits

from github import Auth, Github
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
g    = Github(auth=Auth.Token(GITHUB_TOKEN))
repo = g.get_repo(GITHUB_REPO)

def push_csv(df, filename, commit_message):
    csv_content = df.to_csv(index=False)
    path = f"outputs/{filename}"
    try:
        existing = repo.get_contents(path, ref=GITHUB_BRANCH)
        repo.update_file(path, commit_message, csv_content, existing.sha, branch=GITHUB_BRANCH)
        print(f"✅ Updated {path}")
    except Exception:
        repo.create_file(path, commit_message, csv_content, branch=GITHUB_BRANCH)
        print(f"✅ Created {path}")

push_csv(df_scored,     "df_scored.csv",     "Update df_scored")
push_csv(df_wallet_agg, "df_wallet_agg.csv", "Update df_wallet_agg")
push_csv(df_combined,   "df_combined.csv",   "Update df_combined")

print(f"\nDone — outputs at https://github.com/{GITHUB_REPO}/tree/{GITHUB_BRANCH}/outputs")

✅ Updated outputs/df_scored.csv
✅ Updated outputs/df_wallet_agg.csv
✅ Updated outputs/df_combined.csv

Done — outputs at https://github.com/chadallen/insider_trading_detection/tree/main/outputs


# Debug

In [9]:
verify_sql = """
SELECT DISTINCT question, MIN(block_time) AS first_trade, MAX(block_time) AS last_trade
FROM polymarket_polygon.market_trades
WHERE question LIKE '%Israel%Iran%'
   OR question LIKE '%strike%Iran%'
GROUP BY question
ORDER BY last_trade DESC
"""

exec_id = execute_sql(verify_sql)
print(f"Execution ID: {exec_id}")
if poll_until_done(exec_id):
    df = fetch_results(exec_id)
    print(df[["question", "first_trade", "last_trade"]].to_string())

Execution ID: 01KKCF7Q9TD86PZA6XZZM0HN8V
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting

In [ ]:
test_sql = """
SELECT COUNT(*) AS n
FROM polymarket_polygon.market_trades
WHERE question LIKE '%Maduro%'
  AND block_time BETWEEN TIMESTAMP '2026-01-28' AND TIMESTAMP '2026-01-31'
"""

exec_id = execute_sql(test_sql)
print(f"Execution ID: {exec_id}")
if poll_until_done(exec_id):
    df = fetch_results(exec_id)
    print(df)

Execution ID: 01KKA1S4RSMSRD3FS453BCMDA7
  Status: QUERY_STATE_EXECUTING — waiting 5s...
      n
0  8155
